# 04 · Out-of-sample validation

An explicit development / out-of-sample date split. The OOS window is not
loaded until the development analysis is finished.

**This notebook is the prototype of the Phase 5 study record.** Keep its
structure clean enough to formalize later.

> **This notebook is a research client, not a pipeline.** It contains no SQL, no
> threshold, and no classification rule. Every number comes from
> `afterhours_lab.research`, which reads persisted features computed once by
> `afterhours_lab.reactions`. Nothing here writes to the database — the pool is
> opened read-only.

In [ ]:
import datetime as dt

from afterhours_lab.research import (
    EventFilter,
    fetch_cohort,
    fetch_event_detail,
    fetch_class_distribution,
    fetch_monthly_counts,
    to_csv,
    to_jsonl,
    to_pandas,
    to_polars,
    write_parquet,
)
from afterhours_lab.research.notebook import (
    research_pool,
    describe_filter,
    describe_cohort,
    show_cohort,
    development_split,
)

# Jupyter already runs an event loop, so `await` works at cell top level.
pool = await research_pool()

## Hypothesis

_State the hypothesis in one sentence before looking at anything._ Example:
"Reactions detected within the first few minutes retain more of the initial
move over 105 minutes than reactions detected later."

## Universe and split

In [ ]:
date_from = dt.date.today() - dt.timedelta(days=540)
date_to = dt.date.today() - dt.timedelta(days=1)
(dev_from, dev_to), (oos_from, oos_to) = development_split(date_from, date_to, holdout_fraction=0.3)
print('development:', dev_from, '->', dev_to)
print('out-of-sample:', oos_from, '->', oos_to)

In [ ]:
dev_filter = EventFilter(
    date_from=dev_from, date_to=dev_to,
    analysis_statuses=('complete',), limit=5000,
)
async with pool.acquire() as conn:
    dev_cohort = show_cohort(await fetch_cohort(conn, dev_filter))
dev_df = to_pandas(dev_cohort.rows)

## Development analysis

Only the development cohort is touched here. Split detection delay at its
in-sample median and compare retention.

In [ ]:
import numpy as np
sub = dev_df.dropna(subset=['detection_delay_minutes', 'retention'])
cut = sub['detection_delay_minutes'].median()
early = sub[sub['detection_delay_minutes'] <= cut]['retention']
late = sub[sub['detection_delay_minutes'] > cut]['retention']
print(f'median detection delay (in-sample): {cut:.0f} min')
print(f'early: n={len(early)}  median retention={early.median():.3f}')
print(f'late:  n={len(late)}  median retention={late.median():.3f}')
dev_effect = early.median() - late.median()
print(f'development effect (early - late): {dev_effect:+.3f}')

## Freeze the rule

Write down here — before loading OOS — the exact rule to be tested, using
the in-sample cut value above. The cut is now a constant, not re-fit.

## Out-of-sample check

Now, and not before, load the held-out window and apply the frozen rule once.

In [ ]:
oos_filter = EventFilter(
    date_from=oos_from, date_to=oos_to,
    analysis_statuses=('complete',), limit=5000,
)
async with pool.acquire() as conn:
    oos_cohort = show_cohort(await fetch_cohort(conn, oos_filter))
oos_df = to_pandas(oos_cohort.rows)

In [ ]:
sub = oos_df.dropna(subset=['detection_delay_minutes', 'retention'])
early = sub[sub['detection_delay_minutes'] <= cut]['retention']
late = sub[sub['detection_delay_minutes'] > cut]['retention']
print(f'early: n={len(early)}  median retention={early.median():.3f}')
print(f'late:  n={len(late)}  median retention={late.median():.3f}')
oos_effect = early.median() - late.median()
print(f'OOS effect (early - late): {oos_effect:+.3f}   (development was {dev_effect:+.3f})')

## Study record

The fields a Phase 5 `study_versions` / `study_results` row will need,
filled in from this run:

- **hypothesis** — (above)
- **universe & date range** — after-close events, `date_from`..`date_to`
- **detector / classifier versions** — printed in the cohort disclosure
- **filters** — `dev_filter` / `oos_filter` (Phase 5 stores these as the
  canonical query string from `web/params.filter_to_query`)
- **development / OOS periods** — `dev_from..dev_to` / `oos_from..oos_to`
- **execution assumptions** — none; this is an observational study
- **data snapshot** — Phase 5 digest of the cohort's
  `(symbol, earnings_date, source_evidence_sha256)` triples
- **results** — `dev_effect`, `oos_effect`
- **failure modes** — small OOS n, regime change across the split, look-ahead if the cut were re-fit
- **decision** — reject / refine / paper-trade candidate

In [ ]:
await pool.close()